In [1]:
from doe2vec import doe_model
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import scipy.stats as stats
from sklearn.ensemble import RandomForestClassifier
from Benchmarks import get_dataset
import joblib

In [2]:
def test(D2V,rt):
    a=np.sum(D2V,axis=1)!=0
    b=np.sum(np.isnan(D2V),axis=1)==0
    c=np.sum(np.isinf(D2V),axis=1)==0
    d=np.sum(np.abs(D2V)>10**8,axis=1)==0
    e=np.sum(rt,axis=1)!=0
    ind=a & b & c & d & e
    lRT=rt[ind,:]
    D2V=D2V[ind,:]
    
    lERT=np.zeros([len(lRT),10])
    for i in range(len(lRT)):
        for j in range(10):
            lERT[i,j]=(np.sum(lRT[i,j*30:(j+1)*30])+np.sum(lRT[i,j*30:(j+1)*30]==-1)*10001)/np.max([np.sum(lRT[i,j*30:(j+1)*30]!=-1),1])

    y = np.argmin(lERT, axis=1)
    label = np.zeros([len(y), 301])
    label[:, 0] = y
    lRT[lRT == -1] = 10000
    label[:, 1:] = lRT
    
    (TrainX, TestX,label1, label2) = train_test_split(D2V, label, test_size=0.2,random_state=42)
    TrainY=label1[:,0]
    TrainR=label1[:,1:]
    TestY=label2[:,0]
    TestR=label2[:,1:]
    
    print('here')
    # 创建随机森林分类器
    rf_classifier = RandomForestClassifier(random_state=42)
    # 训练模型
    rf_classifier.fit(TrainX, TrainY)
    y_hat = rf_classifier.predict(TestX)
    
    acc=0
    for i in range(len(TestY)):
        if TestY[i]==y_hat[i]:
            acc+=1
        elif stats.ranksums(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)],TestR[i,int(TestY[i]*30):int((TestY[i]+1)*30)])[1] > 0.05:
            # if np.sum(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)]==10000)<25:
            acc+=1
    print("可接受率",acc/len(TestY))
    joblib.dump(rf_classifier, 'D2V_model.pkl')
    return rf_classifier

In [3]:
#MA
MA_rt = np.zeros([1, 300])
for i in range(20):
    MA_rt = np.append(MA_rt, np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\MA-BBOB\MA-BBOB\\MA-BBOB-{}.csv".format(i), "r"),delimiter=",")[:, 301:601], axis=0)
D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\MA_VAE.csv",delimiter=",")
test(D2V,MA_rt[1:])


here
可接受率 0.833248342682305


RandomForestClassifier(random_state=42)

In [4]:
#RGI
RGI_rt = np.zeros([1, 300])
for i in range(25):
    RGI_rt = np.append(RGI_rt, (np.loadtxt(open(r"D:\Pythonnnnnn\python\MyWork\EXP\MyDataset\Dataset_RT\RT-{}.csv".format(i + 1), "r"),
        delimiter=","))[:, 1:], axis=0)
D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\RGI_VAE.csv",delimiter=",")[:,1:]
test(D2V,RGI_rt)


here


KeyboardInterrupt: 

In [3]:
##  BBOB

BBOB_rt = np.zeros([1, 300])
for i in range(24):
    BBOB_rt = np.append(BBOB_rt,np.loadtxt(open("D:\Dataset\Mywork\DataAll\BBOB\\bbob-{}.csv".format(i + 1), "r"),delimiter=",")[:, 300:600], axis=0)
D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\BBOB_VAE.csv",delimiter=",")
BBOB_rt=BBOB_rt[1:,:]
test(D2V,BBOB_rt)


here
可接受率 0.8044943820224719


RandomForestClassifier(random_state=42)

In [18]:
# Affine
Affine= np.loadtxt(open(r"D:\Dataset\Mywork\DataAll\Affine\affine.csv", "r"), delimiter=",")
Affine_rt = Affine[:, 300:600]
D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\Affine_VAE.csv",delimiter=",")
test(D2V,Affine_rt)

here
可接受率 0.8831578947368421


In [14]:
#Zigzag

Zigzag_rt = np.zeros([1, 300])
for i in range(5):
    for j in range(5):

        Zigzag_rt = np.append(Zigzag_rt, np.loadtxt(
            open("D:\Dataset\Mywork\DataAll\Zigzag\\zigzag-{}-{}.csv".format(i, j), "r"), delimiter=",")[:,300:600], axis=0)
Zigzag_rt=Zigzag_rt[1:,:]
D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\Zigzag_VAE.csv",delimiter=",")
test(D2V,Zigzag_rt)

here
可接受率 0.998


In [21]:
#GPB

GPB_rt = np.zeros([1, 300])
for i in range(5):

    GPB_rt = np.append(GPB_rt, np.loadtxt(open("D:\Dataset\Mywork\DataAll\GPB\\lfA-{}.csv".format(i), "r"),
                                          delimiter=",")[:, 300:600], axis=0)
for i in range(5):

    GPB_rt = np.append(GPB_rt, np.loadtxt(open("D:\Dataset\Mywork\DataAll\GPB\\lfB-{}.csv".format(i), "r"),
                                          delimiter=",")[:, 300:600], axis=0)

D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\GPB_VAE.csv",delimiter=",")
test(D2V,GPB_rt)


here
可接受率 0.6636771300448431


In [16]:
# RGF
RGF_label = np.zeros([1])
for i in range(20):
    RGF_label = np.append(RGF_label, np.loadtxt(
        open("D:\Dataset\Mywork\DataAll\RGF\RGF_ELA\lf-{}.csv".format(i + 1), "r"), delimiter=",")[:, 0],
                          axis=0)
D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\RGF_VAE.csv",delimiter=",")[:,1:]
a = np.sum(D2V, axis=1) != 0
b = np.sum(np.isnan(D2V), axis=1) == 0
c = np.sum(np.isinf(D2V), axis=1) == 0
d = np.sum(np.abs(D2V) > 10 ** 8, axis=1) == 0
ind = a & b & c & d
D2V = D2V[ind, :]
RGF_label = RGF_label[ind]

(TrainX, TestX,TrainY, TestY) = train_test_split(D2V, RGF_label, test_size=0.2,random_state=42)

print('here')
# 创建随机森林分类器
rf_classifier = RandomForestClassifier(random_state=42)
# 训练模型
rf_classifier.fit(TrainX, TrainY)
y_hat = rf_classifier.predict(TestX)


print(np.sum(TestY==y_hat)/len(TestY))

here
0.7473690883732724


%history
from doe2vec import doe_model
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import scipy.stats as stats
from sklearn.ensemble import RandomForestClassifier
%history


In [6]:
##  BBOB VAE+ELA
BBOB_ela = np.zeros([1, 61])
BBOB_rt = np.zeros([1, 300])
for i in range(24):
    BBOB_ela = np.append(BBOB_ela,
                         np.loadtxt(open("D:\Dataset\Mywork\DataAll\BBOB\\bbob-{}.csv".format(i + 1), "r"),
                                    delimiter=",")[:, 600:], axis=0)
    BBOB_rt = np.append(BBOB_rt,
                        np.loadtxt(open("D:\Dataset\Mywork\DataAll\BBOB\\bbob-{}.csv".format(i + 1), "r"),
                                   delimiter=",")[:, 300:600], axis=0)
BBOB_ela=BBOB_ela[1:]
BBOB_rt=BBOB_rt[1:]
D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\BBOB_VAE.csv",delimiter=",")
D2V=np.hstack((BBOB_ela,D2V))
test(D2V,BBOB_rt)

here
可接受率 0.8988235294117647


In [7]:
Affine= np.loadtxt(open(r"D:\Dataset\Mywork\DataAll\Affine\affine.csv", "r"), delimiter=",")
Affine_ela = Affine[:, 600:]
Affine_rt = Affine[:, 300:600]
        
D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\Affine_VAE.csv",delimiter=",")
D2V=np.hstack((Affine_ela,D2V))
test(D2V,Affine_rt)

here
可接受率 0.9122401847575058


In [9]:
#MA
MA_ela = np.zeros([1, 61])
MA_rt = np.zeros([1, 300])
for i in range(20):
    MA_ela = np.append(MA_ela,
                         np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\MA-BBOB\MA-BBOB\\MA-BBOB-{}.csv".format(i), "r"),delimiter=",")[:, 601:], axis=0)
    MA_rt = np.append(MA_rt,
                        np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\MA-BBOB\MA-BBOB\\MA-BBOB-{}.csv".format(i), "r"),delimiter=",")[:, 301:601], axis=0)
MA_ela=MA_ela[1:]
MA_rt=MA_rt[1:]

D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\MA_VAE.csv",delimiter=",")
D2V=np.hstack((MA_ela,D2V))
test(D2V,MA_rt)


here
可接受率 0.8536460989291178


In [11]:
GPB_ela = np.zeros([1, 61])
GPB_rt = np.zeros([1, 300])
for i in range(5):
    GPB_ela = np.append(GPB_ela, np.loadtxt(open("D:\Dataset\Mywork\DataAll\GPB\\lfA-{}.csv".format(i), "r"),
                                            delimiter=",")[:, 600:], axis=0)
    GPB_rt = np.append(GPB_rt, np.loadtxt(open("D:\Dataset\Mywork\DataAll\GPB\\lfA-{}.csv".format(i), "r"),
                                          delimiter=",")[:, 300:600], axis=0)
for i in range(5):
    GPB_ela = np.append(GPB_ela, np.loadtxt(open("D:\Dataset\Mywork\DataAll\GPB\\lfB-{}.csv".format(i), "r"),
                                            delimiter=",")[:, 600:], axis=0)
    GPB_rt = np.append(GPB_rt, np.loadtxt(open("D:\Dataset\Mywork\DataAll\GPB\\lfB-{}.csv".format(i), "r"),
                                          delimiter=",")[:, 300:600], axis=0)

D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\GPB_VAE.csv",delimiter=",")
D2V=np.hstack((GPB_ela,D2V))
test(D2V,GPB_rt)

here
可接受率 0.7899543378995434


In [9]:
RGF_ela = np.zeros([1, 61])
RGF_label = np.zeros([1])
for i in range(20):
    RGF_ela = np.append(RGF_ela, np.loadtxt(
        open("D:\Dataset\Mywork\DataAll\RGF\RGF_ELA\lf-{}.csv".format(i + 1), "r"), delimiter=",")[:, 1:],
                        axis=0)
    RGF_label = np.append(RGF_label, np.loadtxt(
        open("D:\Dataset\Mywork\DataAll\RGF\RGF_ELA\lf-{}.csv".format(i + 1), "r"), delimiter=",")[:, 0],
                          axis=0)
D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\RGF_VAE.csv",delimiter=",")
D2V=np.hstack((RGF_ela,D2V))

a = np.sum(D2V, axis=1) != 0
b = np.sum(np.isnan(D2V), axis=1) == 0
c = np.sum(np.isinf(D2V), axis=1) == 0
d = np.sum(np.abs(D2V) > 10 ** 8, axis=1) == 0
ind = a & b & c & d
D2V = D2V[ind, :]
RGF_label = RGF_label[ind]

(TrainX, TestX,TrainY, TestY) = train_test_split(D2V, RGF_label, test_size=0.2,random_state=1)

print('here')
# 创建随机森林分类器
rf_classifier = RandomForestClassifier(random_state=42)
# 训练模型
rf_classifier.fit(TrainX, TrainY)
y_hat = rf_classifier.predict(TestX)


print(np.sum(TestY==y_hat)/len(TestY))

here
0.8058858699262605


In [10]:
RGI_ela = np.zeros([1, 61])
RGI_rt = np.zeros([1, 300])
for i in range(25):
    RGI_ela = np.append(RGI_ela, (pd.read_csv(
        "D:\Pythonnnnnn\python\MyWork\EXP\MyDataset\Dataset_ELA\ELA-{}.csv".format(i + 1), header=None,
        skip_blank_lines=False).fillna(0).to_numpy())[:, 1:], axis=0)
    RGI_rt = np.append(RGI_rt, (np.loadtxt(
        open("D:\Pythonnnnnn\python\MyWork\EXP\MyDataset\Dataset_RT\RT-{}.csv".format(i + 1), "r"),
        delimiter=","))[:, 1:], axis=0)
D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\RGI_VAE.csv",delimiter=",")
D2V=np.hstack((RGI_ela,D2V))
test(D2V,RGI_rt)

D:\Python39\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


here
可接受率 0.7224682531213318


In [7]:
RGI_ela = np.zeros([1, 61])
RGI_rt = np.zeros([1, 300])
for i in range(25):
    RGI_ela = np.append(RGI_ela, (pd.read_csv(
        "D:\Pythonnnnnn\python\MyWork\EXP\MyDataset\Dataset_ELA\ELA-{}.csv".format(i + 1), header=None,
        skip_blank_lines=False).fillna(0).to_numpy())[:, 1:], axis=0)
    RGI_rt = np.append(RGI_rt, (np.loadtxt(
        open("D:\Pythonnnnnn\python\MyWork\EXP\MyDataset\Dataset_RT\RT-{}.csv".format(i + 1), "r"),
        delimiter=","))[:, 1:], axis=0)
D2V=np.loadtxt(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\D2Vae\RGI_VAE.csv",delimiter=",")
D2V=np.hstack((RGI_ela,D2V))

a=np.sum(D2V,axis=1)!=0
b=np.sum(np.isnan(D2V),axis=1)==0
c=np.sum(np.isinf(D2V),axis=1)==0
d=np.sum(np.abs(D2V)>10**8,axis=1)==0
e=np.sum(RGI_rt,axis=1)!=0
ind=a & b & c & d & e
lRT=RGI_rt[ind,:]
D2V=D2V[ind,:]

lERT=np.zeros([len(lRT),10])
for i in range(len(lRT)):
    for j in range(10):
        lERT[i,j]=(np.sum(lRT[i,j*30:(j+1)*30])+np.sum(lRT[i,j*30:(j+1)*30]==-1)*10001)/np.max([np.sum(lRT[i,j*30:(j+1)*30]!=-1),1])

y = np.argmin(lERT, axis=1)
label = np.zeros([len(y), 301])
label[:, 0] = y
lRT[lRT == -1] = 10000
label[:, 1:] = lRT



KeyboardInterrupt: 

In [9]:
import lightgbm as lgb
(TrainX, TestX,label1, label2) = train_test_split(D2V, label, test_size=0.2,random_state=42)
TrainY=label1[:,0]
TrainR=label1[:,1:]
TestY=label2[:,0]
TestR=label2[:,1:]

print('here')
# 创建随机森林分类器
train_data = lgb.Dataset(TrainX, label=TrainY)
test_data = lgb.Dataset(TestX, label=TestY)

evals_result = {}
params = {
    'objective': 'multiclass',
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'max_depth': 5,
    'num_class': 10,
    'lambda_l1':0.1,
    'lambda_l2':0.1,
    "early_stopping_round": 50,
    "verbose_eval": 10,
    'verbosity': -1,
    # 'num_leaves':10,
    # "device_type": 'GPU'
}
num_round = 2000  # 迭代次数
callbacks= [lgb.log_evaluation(period=1), ]

model = lgb.train(params, train_data, num_round,valid_sets=[train_data,test_data],callbacks=callbacks)

y_pred = model.predict(TestX)
y_hat = np.argmax(y_pred,axis=1)


acc=0
for i in range(len(TestY)):
    if TestY[i]==y_hat[i]:
        acc+=1
    elif stats.ranksums(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)],TestR[i,int(TestY[i]*30):int((TestY[i]+1)*30)])[1] > 0.05:
        # if np.sum(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)]==10000)<25:
        acc+=1
print("可接受率",acc/len(TestY))

here
[1]	training's multi_logloss: 1.91307	valid_1's multi_logloss: 1.91713
[2]	training's multi_logloss: 1.86806	valid_1's multi_logloss: 1.87319
[3]	training's multi_logloss: 1.82984	valid_1's multi_logloss: 1.836
[4]	training's multi_logloss: 1.79674	valid_1's multi_logloss: 1.80375
[5]	training's multi_logloss: 1.76782	valid_1's multi_logloss: 1.77566
[6]	training's multi_logloss: 1.7418	valid_1's multi_logloss: 1.75049
[7]	training's multi_logloss: 1.71863	valid_1's multi_logloss: 1.72806
[8]	training's multi_logloss: 1.69776	valid_1's multi_logloss: 1.70794
[9]	training's multi_logloss: 1.67888	valid_1's multi_logloss: 1.68982
[10]	training's multi_logloss: 1.66185	valid_1's multi_logloss: 1.67354
[11]	training's multi_logloss: 1.64609	valid_1's multi_logloss: 1.6585
[12]	training's multi_logloss: 1.6319	valid_1's multi_logloss: 1.64512
[13]	training's multi_logloss: 1.61882	valid_1's multi_logloss: 1.63285
[14]	training's multi_logloss: 1.60692	valid_1's multi_logloss: 1.62164
[